# Pasajeros · Segmentación de personas dentro del vehículo (punto adicional)
Se usa YOLO11-seg (clase `person`) y se cuentan las personas cuyo centro cae dentro de la caja del vehículo detectado por M2.
**No** hay reconocimiento facial ni se guardan recortes: la evidencia cubre cada persona con una máscara opaca.

1. Métricas del modelo de segmentación (COCO val, clase persona) → `yolo val`.
2. Error de conteo en fotos propias: subir `pasajeros_eval/` con fotos y `conteos.txt` (`foto1.jpg=3`).

In [ ]:
# Versiones fijadas (mismas que inferencia/requirements.txt)
!pip install -q ultralytics==8.3.240 roboflow==1.1.66 "pillow-heif<1.0"

In [ ]:
# Diagnóstico del entorno (punto adicional: gestión de entornos). Guardar la salida en la carpeta de evidencias.
import sys, platform, subprocess
import torch, ultralytics
print("python:", sys.version.split()[0])
print("sistema:", platform.platform())
print("torch:", torch.__version__, "| cuda:", torch.version.cuda, "| gpu disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("ultralytics:", ultralytics.__version__)
subprocess.run("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv", shell=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SALIDA = '/content/drive/MyDrive/peaje_evidencias'
import os; os.makedirs(SALIDA, exist_ok=True)

In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolo11n-seg.pt')
# Validación en COCO 2017 val (5000 imágenes, ~1 GB). Métricas de la clase persona (índice 0).
m = modelo.val(data='coco.yaml', classes=[0], imgsz=640, batch=32)
print(f'persona: máscara mAP50={m.seg.map50:.3f} mAP50-95={m.seg.map:.3f} | caja mAP50={m.box.map50:.3f}')

In [ ]:
# Error de conteo en fotos propias de vehículos con pasajeros visibles.
import os, numpy as np
CARPETA = '/content/pasajeros_eval'
conteos = dict(l.strip().split('=') for l in open(f'{CARPETA}/conteos.txt') if '=' in l)
errores = []
for nombre, real in conteos.items():
    r = modelo.predict(f'{CARPETA}/{nombre}', classes=[0], conf=0.35, verbose=False)[0]
    pred = len(r.boxes)
    errores.append(abs(pred - int(real)))
    print(nombre, 'real', real, 'pred', pred)
print(f'MAE pasajeros: {np.mean(errores):.2f} | exactos: {np.mean(np.array(errores) == 0):.1%}')